In [ ]:
import apache_beam as beam
import os
import json
import importlib
import unittest

from datetime import datetime

In [2]:
fromNotebook = True
configFile = "load_wwi.json"
newCutoffDate = "2013-01-01"

In [3]:
prefix = ""
if fromNotebook == False:
    prefix = "notebooks/"

file_path = os.path.join(os.getcwd(), f"{prefix}modules", 'sql_utilities.py')
spec = importlib.util.spec_from_file_location("sql_utils", file_path)
sql_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_utils)

In [4]:
f = open(configFile,)
config = json.load(f)
f.close()

tables = config["tables"]
source_db = config["source"]
destination_db = config["destination"]
initialCutoffDate = config["initialCutoffDate"]
if fromNotebook == True:
    newCutoffDate = config["cutoffDate"]
load_tables = [
    {
        "name": table["name"],
        "schema": table["destination"]["schema"],
        "table": table["destination"]["table"]
    }
    for table in tables
]

print(f"tables: {tables}")
print(f"source_db: {source_db}")
print(f"destination_db: {destination_db}")
print(f"initialCutoffDate: {initialCutoffDate}")
print(f"newCutoffDate: {newCutoffDate}")
print(f"load_tables: {load_tables}")


tables: [{'name': 'ApplicationCities', 'source': {'schema': 'Application', 'table': 'Cities', 'loadTable': 'scripts/load_wwi_mssql/load_application_cities_2013-01-01.sql'}, 'destination': {'schema': 'dbo', 'table': 'Application_Cities', 'stagingTable': 'Application_Cities_Staging', 'createTable': 'scripts/load_wwi_mssql/create_application_cities.sql', 'modifyTable': [], 'insertTable': 'scripts/load_wwi_mssql/insert_record.sql', 'dropTable': 'scripts/load_wwi_mssql/drop_table.sql', 'deleteByLoadDateTable': 'scripts/load_wwi_mssql/delete_record_by_load_date.sql', 'specificColumnTypes': [{'name': 'Location', 'type': 'VARBINARY(MAX)', 'typeCast': 'CAST(? AS VARBINARY(MAX))'}], 'testLoadTable': 'scripts/load_wwi_mssql/test/load_application_cities_2013-01-01.sql', 'testCountTable': 'scripts/load_wwi_mssql/test/validrange_record_count.sql'}}, {'name': 'ApplicationCitiesArchive', 'source': {'schema': 'Application', 'table': 'Cities_Archive', 'loadTable': 'scripts/load_wwi_mssql/load_applicatio

In [5]:
class LoadData(beam.DoFn):
    def __init__(self, database_type, conn_str, load_path, values_to_replace, tables_to_replace):
        self.database_type = database_type
        self.conn_str = conn_str
        self.load_path = load_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            self.load_path, 
            self.values_to_replace, 
            self.tables_to_replace
        )
        
        return sql_utils.select_sql(self.conn_str, sql, self.database_type)

In [6]:
class WriteData(beam.DoFn):
    def __init__(self, database_type, conn_str, insert_type, insert_path, values_to_replace, specific_column_types, excluded_columns = [], yield_record = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.insert_type = insert_type
        self.insert_path = insert_path
        self.values_to_replace = values_to_replace
        self.specific_column_types = specific_column_types
        self.excluded_columns = excluded_columns
        self.yield_record = yield_record

    def start_bundle(self):
        self.batch = []
        self.columns = []
        
    def process(self, element):
        if len(self.columns) == 0:
            self.columns = list(element.keys())
            for excluded_column in self.excluded_columns:
                self.columns.remove(excluded_column)

        row_values = ()

        for column in self.columns:
            row_values += (element[column],)
        
        if self.insert_type == "batch":
            self.batch.append(row_values)
        else:
            sql = self.get_sql()
            
            sql_utils.exec_sql(
                self.conn_str,
                sql,
                row_values,
                self.database_type
            )
        
        if self.yield_record == True:
            yield element

    def get_sql(self):
        values = []

        for column in self.columns:
            excluded_column = next((item for item in self.specific_column_types if item['name'] == column), None)           

            if excluded_column is None:
                values.append("?")
            else:
                values.append(excluded_column["typeCast"])

        sql = sql_utils.get_sql_from_script(
            self.insert_path, 
            self.values_to_replace, 
            []
        )

        sql = sql_utils.replace_sql_values(
            sql, 
            [
                { 
                    "name": "Columns",
                    "value": ",".join(self.columns)
                },
                {
                    "name": "ColumnValues",
                    "value": ",".join(values)
                }
            ]
        )

        return sql

    def finish_bundle(self):
        if len(self.batch) > 0:
            sql = self.get_sql()
            sql_utils.exec_sql_batch(self.conn_str, sql, self.batch, self.database_type)

In [7]:
class UpsertData(beam.DoFn):
    def __init__(self, database_type, conn_str, upsert_path, values_to_replace, tables_to_replace, yield_record = None, show_sql = False, yield_element = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.upsert_path = upsert_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.yield_record = yield_record
        self.show_sql = show_sql
        self.yield_element = yield_element

    def process(self, element):
        sql = sql_utils.get_sql_from_script(
            self.upsert_path, 
            self.values_to_replace, 
            self.tables_to_replace
        )

        if self.show_sql == True:
            print(f"""Upsert Data sql:
                  {sql} 
                  """)
            
            print(f"element: {element}")
            print(f"len element: {len(element)}")
        
        sql_utils.exec_sql(self.conn_str, sql, None, self.database_type)

        if self.yield_record is not None:
            yield self.yield_record
        else:
            if self.yield_element == True:
                yield element

In [8]:
tc = unittest.TestCase()

class AssertData(beam.DoFn):
    def __init__(self, database_type, conn_str, load_path, count_path, values_to_replace, tables_to_replace, test, test_record_count, yield_record = None, print_sql = False):
        self.database_type = database_type
        self.conn_str = conn_str
        self.load_path = load_path
        self.count_path = count_path
        self.values_to_replace = values_to_replace
        self.tables_to_replace = tables_to_replace
        self.test = test
        self.test_record_count = test_record_count
        self.yield_record = yield_record
        self.print_sql = print_sql

    def process(self, element):
        if self.test == True:
            count_sql = sql_utils.get_sql_from_script(
                self.count_path,
                self.values_to_replace, 
                self.tables_to_replace
            )

            expected_data_sql = sql_utils.get_sql_from_script(
                self.load_path, 
                self.values_to_replace, 
                self.tables_to_replace
            )

            if self.print_sql == True:
                print(f"count_sql: {count_sql}")
                print(f"expected_data_sql: {expected_data_sql}")

            expected_data_count = sql_utils.exec_sql_scalar(self.conn_str, count_sql, self.database_type)
            expected_data_sample = list(sql_utils.select_sql(self.conn_str, expected_data_sql, self.database_type))
            
            input_data_count = len(element)
            input_data_sample = element[:self.test_record_count]
            
            tc.assertEqual(input_data_count, expected_data_count)
            tc.assertListEqual(input_data_sample, expected_data_sample)

        if self.yield_record is not None:
            yield self.yield_record

In [9]:
p = beam.Pipeline()

def get_date_values(last_catuoff_date, new_cutoff_date):
    return [
        { "name": "LastCutoffDate", "value": last_catuoff_date },
        { "name": "NewCutoffDate", "value": new_cutoff_date }
    ]

def get_last_cutoff_date(table_name):
    lastCutoffDateFromLoadHistory = sql_utils.exec_sql_scalar(
        destination_db["conn"],
        sql_utils.get_sql_from_script(
            config["loadHistory"]["loadLastCutoffDateLoadHistoryTable"], 
            [
                { "name": "TableName", "value": table_name },
                { "name": "Schema", "value": config["loadHistory"]["schema"] },
                { "name": "Table", "value": config["loadHistory"]["table"] }
            ],
            []
        ),
        destination_db["databaseType"]
    )

    if lastCutoffDateFromLoadHistory is None:
        return initialCutoffDate
    else:
        return lastCutoffDateFromLoadHistory.strftime("%Y-%m-%d %H:%M:%S")

def modify_table(table):
    tableName = table["destination"]["table"]
    lastCutoffDate = get_last_cutoff_date(tableName)
    print(f"modify_table lastCutoffDate: {lastCutoffDate}")

    modifyTableScripts = []
                
    for modifyTableScript in table["destination"]["modifyTable"]:
        modifyTableScripts.append(
            sql_utils.get_sql_from_script(
                config["modifyLoadHistory"]["loadUnprocessedModifyLoadHistoryScriptTable"],
                [
                    { "name": "ScriptName", "value": modifyTableScript["tableUpdate"] },
                    { "name": "LoadScriptName", "value": modifyTableScript["tableDataLoad"] },
                    { "name": "UpdateScriptName", "value": modifyTableScript["tableDataUpdate"] },
                    { "name": "TestScriptName", "value": modifyTableScript["tableDataTest"] },
                    { "name": "TestCountScriptName", "value": modifyTableScript["tableCountDataTest"] },
                    { "name": "CreateStagingScriptName", "value": modifyTableScript["tableCreateStaging"] },
                    { "name": "Name", "value": modifyTableScript["name"] }
                ]
            )
        )
    
    unprocessedScripts = sql_utils.select_sql(
        destination_db["conn"],
        sql_utils.get_sql_from_script(
            config["modifyLoadHistory"]["loadUnprocessedModifyLoadHistoryTable"],
            [
                { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                { "name": "Table", "value": config["modifyLoadHistory"]["table"] },
                { "name": "TableName", "value": tableName },
                { "name": "LoadDate", "value": lastCutoffDate },
                { "name": "ModifyTableScripts", "value": " UNION ".join(modifyTableScripts) }
            ]
        ),
        destination_db["databaseType"]
    )

    input = (
        p
        | f"Modify Create Empty collection for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }]) 
    )

    for script in unprocessedScripts:
        print(f"Processing '{script["Name"]}' for table {tableName} for cutoff {newCutoffDate}.")

        input = (
            input
            | f"Modify Table {tableName}" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        script["ScriptName"],
                        [
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                        ],
                        [],
                        { f"Successfully modified table {tableName}." }
                    )
                )
            | f"Modify Table {tableName} Create Staging Table" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        script["CreateStagingScriptName"],
                        [
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["stagingTable"] },
                        ],
                        [],
                        { f"Successfully modified table {tableName}." }
                    )
                )
            | f"Modify Table {tableName} Load Data" >>
                beam.ParDo(
                    LoadData(
                        source_db["databaseType"],
                        source_db["conn"],
                        script["LoadScriptName"],
                        [
                            { "name": "Schema", "value": table["source"]["schema"] },
                            { "name": "Table", "value": table["source"]["table"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate }
                        ],
                        []
                    )
                )
            | f"Modify Table {tableName} Write Data to Staging" >>
                beam.ParDo(
                    WriteData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        "batch",
                        table["destination"]["insertTable"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["stagingTable"] }
                        ],
                        table["destination"]["specificColumnTypes"],
                        [],
                        True
                    )            
                )
            | f"Modify Table {tableName} Combined List for Table" >> 
                beam.combiners.ToList()        
            | f"Modify Table {tableName} Update Data" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        script["UpdateScriptName"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                            { "name": "StagingTable", "value": table["destination"]["stagingTable"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate }
                        ],
                        [],
                        None,
                        False,
                        True
                    )
                )
            | f"Modify Table {tableName} Assert for Table" >>
                beam.ParDo(
                    AssertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        script["TestScriptName"],
                        script["TestCountScriptName"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate },
                            { "name": "NewCutoffDate", "value": newCutoffDate },
                            { "name": "NumberOfRows", "value": str(config["testRecordCount"]) }
                        ],
                        [],
                        config["test"],
                        config["testRecordCount"],
                        { f"Successfully asserted table {tableName}." }
                    )
                )
            | f"Modify Table {tableName} Rows Count for Table" >> 
                beam.combiners.Count.Globally()
            | f"Modify Table {tableName} Create History Record" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["modifyLoadHistory"]["insertModifyLoadHistoryTable"],
                        [
                            { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                            { "name": "Table", "value": config["modifyLoadHistory"]["table"] },
                            { "name": "TableName", "value": tableName },
                            { "name": "LoadDate", "value": lastCutoffDate },
                            { "name": "Status", "value": "Successful" },
                            { "name": "Details", "value": "" },
                            { "name": "ScriptName", "value": script["Name"] }
                        ],
                        [],
                        { "PlaceHolder": "Temp 1 record" }
                    )
                )
            | f"Modify Table {tableName} Drop Staging Table" >> 
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        table["destination"]["dropTable"],
                        [
                            { "name": "Database", "value": destination_db["database"] },
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["stagingTable"] },
                        ],
                        [],
                        { f"Successfully modified table {tableName}." }
                    )
                )
             
        )

    return input

# Create Load History and Modify History Table
(
    p 
    | "Input Create Load History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
    | "Create Load History Table" >> 
        beam.ParDo(
            UpsertData(
                destination_db["databaseType"],
                destination_db["conn"],
                config["loadHistory"]["createLoadHistoryTable"],
                [
                    { "name": "Database", "value": config["destination"]["database"] },
                    { "name": "Schema", "value": config["loadHistory"]["schema"] },
                    { "name": "Table", "value": config["loadHistory"]["table"] }
                ],
                [],
                { "PlaceHolder": "Temp 1 record" }
            )
        )
    | "Create Modify Load History Table" >> 
        beam.ParDo(
            UpsertData(
                destination_db["databaseType"],
                destination_db["conn"],
                config["modifyLoadHistory"]["createModifyLoadHistoryTable"],
                [
                    { "name": "Database", "value": config["destination"]["database"] },
                    { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
                    { "name": "Table", "value": config["modifyLoadHistory"]["table"] }
                ],
                [],
                { "PlaceHolder": "Temp 1 record" }
            )
        )
)

p.run().wait_until_finish()

p = beam.Pipeline()

for table in tables:
    tableName = table["destination"]["table"]

    print(f"Processing table '{tableName}' for '{newCutoffDate}'.")

    lastCutoffDate = get_last_cutoff_date(tableName)
    lastCutoffDateVal = datetime.strptime(lastCutoffDate, "%Y-%m-%d %H:%M:%S")
    newCutoffDateVal = datetime.strptime(newCutoffDate, "%Y-%m-%d %H:%M:%S")
    
    if lastCutoffDateVal >= newCutoffDateVal:
        print(f"Table '{tableName}' has already been loaded for '{newCutoffDate}'.")
    else:
        
        input = p | f"Input Create Table for {tableName}" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
        
        # Modify table if there are changes
        if len(table["destination"]["modifyTable"]) > 0:
            input = modify_table(table)

        result = (
            input
            | f"Create Table for {tableName}" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        table["destination"]["createTable"],
                        [
                            { "name": "Database", "value": config["destination"]["database"] },
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] }
                        ],
                        [],
                        { "PlaceHolder": "Temp 1 record" },
                        False
                    )
                )
            | f"Delete Data with the cutoffdate for {tableName}" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        table["destination"]["deleteByLoadDateTable"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                            { "name": "NewCutoffDate", "value": newCutoffDate }
                        ],
                        [],
                        { "PlaceHolder": "Temp 1 record" },
                        False
                    )
                ) 
            | f"Load Data for Table {tableName}" >> 
                beam.ParDo(
                    LoadData(
                        source_db["databaseType"],
                        source_db["conn"],
                        table["source"]["loadTable"],
                        [
                            { "name": "Schema", "value": table["source"]["schema"] },
                            { "name": "Table", "value": table["source"]["table"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate },
                            { "name": "NewCutoffDate", "value": newCutoffDate }
                        ],
                        []
                    )
                )
            | f"Write Data for Table {tableName}" >> 
                beam.ParDo(
                    WriteData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        "batch",
                        table["destination"]["insertTable"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate },
                            { "name": "NewCutoffDate", "value": newCutoffDate }
                        ],
                        table["destination"]["specificColumnTypes"],
                        [],
                        True
                    )
                )
            | f"Combined List for Table {tableName}" >> 
                beam.combiners.ToList()
            | f"Assert for Table {tableName}" >>
                beam.ParDo(
                    AssertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        table["destination"]["testLoadTable"],
                        table["destination"]["testCountTable"],
                        [
                            { "name": "Schema", "value": table["destination"]["schema"] },
                            { "name": "Table", "value": table["destination"]["table"] },
                            { "name": "LastCutoffDate", "value": lastCutoffDate },
                            { "name": "NewCutoffDate", "value": newCutoffDate },
                            { "name": "NumberOfRows", "value": str(config["testRecordCount"]) }
                        ],
                        [],
                        config["test"],
                        config["testRecordCount"]
                    )
                )
            | f"Rows Count for Table {tableName}" >> 
                beam.combiners.Count.Globally()
            | f"Insert Load History Record for Table {tableName}" >>
                beam.ParDo(
                    UpsertData(
                        destination_db["databaseType"],
                        destination_db["conn"],
                        config["loadHistory"]["insertLoadHistoryTable"],
                        [
                            { "name": "TableName", "value": table["destination"]["table"] },
                            { "name": "LoadDate", "value": newCutoffDate },
                            { "name": "Status", "value": "Successful" },
                            { "name": "Details", "value": "" },
                            { "name": "Schema", "value": config["loadHistory"]["schema"] },
                            { "name": "Table", "value": config["loadHistory"]["table"] }
                        ],
                        [],
                        { f"Successfully processed table {tableName} for {newCutoffDate}." },
                        False
                    )
                )
            | f"Result Table {tableName}" >> beam.Map(print)
        )

p.run().wait_until_finish()

Processing table 'Application_Cities' for '2013-01-01 00:00:00'.
Table 'Application_Cities' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_Cities_Archive' for '2013-01-01 00:00:00'.
Table 'Application_Cities_Archive' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_Countries' for '2013-01-01 00:00:00'.
Table 'Application_Countries' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_Countries_Archive' for '2013-01-01 00:00:00'.
Table 'Application_Countries_Archive' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_DeliveryMethods' for '2013-01-01 00:00:00'.
Table 'Application_DeliveryMethods' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_DeliveryMethods_Archive' for '2013-01-01 00:00:00'.
Table 'Application_DeliveryMethods_Archive' has already been loaded for '2013-01-01 00:00:00'.
Processing table 'Application_PaymentMethods' 

'DONE'

In [13]:
p = beam.Pipeline()

(
    p 
    | "Input Create Load History Table" >> beam.Create([{ "PlaceHolder": "Temp 1 record" }])
    # | "Create Load History Table" >> 
    #     beam.ParDo(
    #         UpsertData(
    #             destination_db["databaseType"],
    #             destination_db["conn"],
    #             config["loadHistory"]["createLoadHistoryTable"],
    #             [
    #                 { "name": "Database", "value": config["destination"]["database"] },
    #                 { "name": "Schema", "value": config["loadHistory"]["schema"] },
    #                 { "name": "Table", "value": config["loadHistory"]["table"] }
    #             ],
    #             [],
    #             { "PlaceHolder": "Temp 1 record" }
    #         )
    #     )
    # | "Create Modify Load History Table" >> 
    #     beam.ParDo(
    #         UpsertData(
    #             destination_db["databaseType"],
    #             destination_db["conn"],
    #             config["modifyLoadHistory"]["createModifyLoadHistoryTable"],
    #             [
    #                 { "name": "Database", "value": config["destination"]["database"] },
    #                 { "name": "Schema", "value": config["modifyLoadHistory"]["schema"] },
    #                 { "name": "Table", "value": config["modifyLoadHistory"]["table"] }
    #             ],
    #             [],
    #             { "PlaceHolder": "Temp 1 record" }
    #         )
    #     )
)

p.run().wait_until_finish()

AttributeError: type object 'DataFrame' has no attribute 'first'